[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/b_10_remat_checkpoint_solution.ipynb)

# 🟡 Solution: Gradient Checkpointing with jax.checkpoint

*JAX Fundamentals · Medium*

Reference implementation. Try it yourself in `b_10_remat_checkpoint.ipynb` first.

---
Apply a deep stack of blocks with **gradient checkpointing** (rematerialization).

Each block is
$$h \leftarrow \tanh(h W_i)$$

Given `params` (a list of `(D, D)` weight matrices) and `x` of shape `(B, D)`,
apply every block in order and return the final `(B, D)` activations — with each
block wrapped in `jax.checkpoint`.

### Rules
- Wrap **each block**, not the whole chain
- The output and gradients must be **numerically identical** to the unwrapped version
- Use `jax.checkpoint` (a.k.a. `jax.remat`)

### The tradeoff
Reverse-mode autodiff normally keeps every intermediate from the forward pass
alive until the backward pass consumes it. For `L` layers that each stash `k`
tensors internally, that is $O(kL)$.

`jax.checkpoint` says: *don't keep this block's internals — recompute them when
the backward pass gets here.* Checkpointing every layer stores one tensor per
layer **boundary** instead of `k` per layer, so $O(kL)$ becomes $O(L)$.

Be precise about that: it is a large constant-factor win, **not** an asymptotic
one. The boundary activations still grow with depth. The genuinely sublinear
scheme is Chen et al. (2016), *Training Deep Nets with Sublinear Memory Cost*,
which checkpoints every $\sqrt{L}$-th layer for $O(\sqrt{L})$ memory.

The price is one extra forward evaluation of each rematerialized region. A
backward pass already costs roughly 2x a forward pass, so recomputing the
forward once takes the total from about 3x to about 4x — the familiar "~33%
more compute" figure.

### The granularity trap
Wrapping the *whole* chain in one `checkpoint` looks like the aggressive choice
and saves nothing. Remat drops the intermediates during the forward pass, but
the backward pass then has to recompute the entire chain in one go — and that
recomputation materializes all `L` layers' activations anyway. Peak memory is
unchanged and you paid for the extra forward. The memory you get is decided by
how finely you cut the chain.

### Why it matters
This is exactly how large transformers are trained — one `checkpoint` per
transformer layer is standard practice, and it is often the difference between
a model fitting in HBM and not. Expect a follow-up about where the extra compute
comes from, and about `policy=` — e.g.
`jax.checkpoint(f, policy=jax.checkpoint_policies.dots_with_no_batch_dims_saveable)`
keeps the expensive matmul outputs and rematerializes only the cheap elementwise
ops, which recovers most of the memory for a fraction of the recompute.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp


def block(W, h):
    return jnp.tanh(h @ W)


def deep_chain(params, x):
    h = x
    for W in params:
        # Wrap each block: its internals are recomputed in the backward pass
        # instead of being held in memory from the forward pass.
        h = jax.checkpoint(block)(W, h)
    return h

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp

keys = jax.random.split(jax.random.key(0), 4)
params = [jax.random.normal(k, (8, 8)) * 0.5 for k in keys]
x = jax.random.normal(jax.random.key(1), (2, 8))

out = deep_chain(params, x)
print("output shape:", out.shape)

# The remat2 primitive in the gradient's jaxpr is the proof it is checkpointed.
jaxpr = str(jax.make_jaxpr(lambda p, v: jnp.sum(deep_chain(p, v)))(params, x))
print("checkpointed:", "remat" in jaxpr)

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("remat_checkpoint")